# Opportunity Home Move-Out Analysis: Six Key Questions

This notebook uses the provided Opportunity Home move-out files to answer six project questions as well as the current data allows.

Files used:

- `MoveOut Activity FY20-26 and FY15 - Data.csv`
- `Move Out Reasons.csv`
- `PHMoveActivity - Business Metadata and Policy Tags - table schema.csv`
- Optional reference: `CIS internship data .xlsx`

Important limitation: the move-out activity data contains households who moved out or transferred. It does **not** include the full population of households who stayed housed. Therefore, this notebook can analyze patterns among move-out records, but it cannot fully predict whether an active household will move out without additional population-at-risk data.


## 0. Setup


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed. Chart cells will be skipped.")

try:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, roc_auc_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("scikit-learn is not installed. Modeling cells will be skipped unless you install scikit-learn.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [ ]:
# Update these paths if you run this notebook somewhere else.
BASE = Path("/Users/josegranados/Downloads/OHSA Move Out Data - CSV")
ACTIVITY_PATH = BASE / "MoveOut Activity FY20-26 and FY15 - Data.csv"
REASONS_PATH = BASE / "Move Out Reasons.csv"
SCHEMA_PATH = BASE / "PHMoveActivity - Business Metadata and Policy Tags - table schema.csv"
CIS_XLSX_PATH = Path("/Users/josegranados/Downloads/CIS internship data .xlsx")

OUTPUT_DIR = Path("/Users/josegranados/Documents/Opportunity Home/outputs/six_question_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [ACTIVITY_PATH, REASONS_PATH, SCHEMA_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

print("Output folder:", OUTPUT_DIR)


## 1. Load and Prepare Data


In [ ]:
moveouts = pd.read_csv(ACTIVITY_PATH)
reasons = pd.read_csv(REASONS_PATH)
schema = pd.read_csv(SCHEMA_PATH)

# Join official resident outcome labels from the move-out reason dictionary.
df = moveouts.merge(
    reasons[["HouseholdMoveOutReason", "Resident Outcome"]],
    on="HouseholdMoveOutReason",
    how="left"
)

# Convert dates.
for col in ["HouseholdMoveInDate", "HouseholdMoveOutDate", "ImportDate"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Convert numeric columns.
for col in ["HOHAgeAtMoveOut", "HouseholdRentToCharge", "HouseholdUnitTenure", "HouseholdProgramTenure"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Outcome flags.
df["negative_exit"] = np.where(df["Resident Outcome"].eq("Negative"), 1, 0)
df["positive_exit"] = np.where(df["Resident Outcome"].eq("Positive"), 1, 0)
df["neutral_exit"] = np.where(df["Resident Outcome"].eq("Neutral"), 1, 0)
df["voluntary_exit"] = np.where(df["HouseholdMoveOutCategory"].eq("Voluntary Termination"), 1, 0)
df["transfer_exit"] = np.where(df["HouseholdMoveOutCategory"].eq("Transfer"), 1, 0)
df["eviction_exit"] = np.where(df["HouseholdMoveOutCategory"].eq("Evicted"), 1, 0)
df["involuntary_termination_exit"] = np.where(df["HouseholdMoveOutCategory"].eq("Involuntary Termination"), 1, 0)

# Special flags for analysis.
df["criminal_drug_related"] = df["HouseholdMoveOutCategoryDetail"].astype(str).str.contains("Criminal/Drug", case=False, na=False).astype(int)
df["non_payment_related"] = df["HouseholdMoveOutCategoryDetail"].astype(str).str.contains("Non Payment", case=False, na=False).astype(int)
df["self_sufficiency_exit"] = df["HouseholdMoveOutReason"].isin([
    "Vol R-Self Sufficiency/Purchased Home",
    "Vol R-Self Sufficiency/Mkt Rent",
]).astype(int)

print("Move-out rows:", len(df))
print("Columns:", len(df.columns))
df.head()


In [ ]:
# Completeness check.
complete_rows = df[moveouts.columns].notna().all(axis=1).sum()
total_rows = len(df)
print(f"Complete rows across original activity fields: {complete_rows}/{total_rows}")

missing_summary = (
    df[moveouts.columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_percent"] = missing_summary["missing_count"] / total_rows
missing_summary.head(15)


In [ ]:
# Overall outcome and category summary.
outcome_summary = df["Resident Outcome"].value_counts(dropna=False).rename_axis("Resident Outcome").reset_index(name="records")
outcome_summary["share"] = outcome_summary["records"] / len(df)

category_summary = df["HouseholdMoveOutCategory"].value_counts(dropna=False).rename_axis("Move-Out Category").reset_index(name="records")
category_summary["share"] = category_summary["records"] / len(df)

display(outcome_summary)
display(category_summary)


## Question 1. Are there indicators of risk that can be identified?

This section identifies factors associated with **negative exits among households that already moved out**. It does not predict whether active households will move out, because active/non-move-out household records are not included.


In [ ]:
# Descriptive negative-exit rates by key fields.
def rate_table(data, group_col, min_records=1):
    table = (
        data.groupby(group_col, dropna=False)
        .agg(
            records=("HouseholdEntityId", "count"),
            negative_exits=("negative_exit", "sum"),
            positive_exits=("positive_exit", "sum"),
            transfers=("transfer_exit", "sum"),
        )
        .reset_index()
    )
    table["negative_exit_rate"] = table["negative_exits"] / table["records"]
    table["positive_exit_rate"] = table["positive_exits"] / table["records"]
    table["transfer_share"] = table["transfers"] / table["records"]
    return table[table["records"] >= min_records].sort_values("negative_exit_rate", ascending=False)

negative_by_household_type = rate_table(df, "HouseholdFamilyTypeHud")
negative_by_race = rate_table(df, "HOHRace")
negative_by_ethnicity = rate_table(df, "HOHEthnicity")
negative_by_property = rate_table(df, "DevelopmentName", min_records=50)

display(negative_by_household_type)
display(negative_by_race)
display(negative_by_ethnicity)
display(negative_by_property.head(20))


In [ ]:
# Optional logistic regression model for negative exits.
# Avoid predictors that directly define the target, such as reason/category/detail.
model_df = df[df["Resident Outcome"].notna()].copy()

predictors = [
    "HOHAgeAtMoveOut",
    "HouseholdRentToCharge",
    "HouseholdUnitTenure",
    "HouseholdProgramTenure",
    "HouseholdFamilyTypeHud",
    "HOHRace",
    "HOHEthnicity",
    "DevelopmentName",
    "FiscalYear",
    "HouseholdMoveInTransfer",
]

numeric_features = [
    "HOHAgeAtMoveOut",
    "HouseholdRentToCharge",
    "HouseholdUnitTenure",
    "HouseholdProgramTenure",
]

categorical_features = [
    "HouseholdFamilyTypeHud",
    "HOHRace",
    "HOHEthnicity",
    "DevelopmentName",
    "FiscalYear",
    "HouseholdMoveInTransfer",
]

if SKLEARN_AVAILABLE:
    X = model_df[predictors]
    y = model_df["negative_exit"]

    preprocess = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    negative_exit_model = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=2000)),
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    negative_exit_model.fit(X_train, y_train)
    y_pred = negative_exit_model.predict(X_test)
    y_prob = negative_exit_model.predict_proba(X_test)[:, 1]

    print(classification_report(y_test, y_pred))
    print("AUC:", roc_auc_score(y_test, y_prob))
else:
    print("Skipping model: scikit-learn is not installed.")


In [ ]:
# Coefficients for interpreting the negative-exit model.
if SKLEARN_AVAILABLE:
    cat_names = negative_exit_model.named_steps["preprocess"].named_transformers_["cat"].get_feature_names_out(categorical_features)
    feature_names = numeric_features + list(cat_names)
    coefs = negative_exit_model.named_steps["model"].coef_[0]

    negative_exit_coefficients = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
        "odds_ratio": np.exp(coefs),
    }).sort_values("coefficient", ascending=False)

    display(negative_exit_coefficients.head(25))
    display(negative_exit_coefficients.tail(25))
else:
    negative_exit_coefficients = pd.DataFrame()


## Question 2. How did COVID-19 impact exits?

The dataset includes FY 2014-2015 and FY 2020-2021 through FY 2025-2026, but it does **not** include FY 2015-2016 through FY 2019-2020. This means COVID-related findings should be treated as trends or hypotheses, not causal proof.


In [ ]:
covid_summary = (
    df.groupby("FiscalYear")
    .agg(
        total_moveouts=("HouseholdEntityId", "count"),
        negative_exits=("negative_exit", "sum"),
        positive_exits=("positive_exit", "sum"),
        neutral_exits=("neutral_exit", "sum"),
        transfers=("transfer_exit", "sum"),
        evictions=("eviction_exit", "sum"),
        involuntary_terminations=("involuntary_termination_exit", "sum"),
        criminal_drug_related=("criminal_drug_related", "sum"),
        non_payment_related=("non_payment_related", "sum"),
    )
    .reset_index()
)

for col in ["negative_exits", "positive_exits", "neutral_exits", "transfers", "evictions", "involuntary_terminations", "criminal_drug_related", "non_payment_related"]:
    covid_summary[col.replace("_exits", "") + "_rate"] = covid_summary[col] / covid_summary["total_moveouts"]

covid_summary


In [ ]:
if plt is not None:
    plot_df = covid_summary.set_index("FiscalYear")
    ax = plot_df[["negative_rate", "transfer_rate", "evictions_rate", "positive_rate"]].plot(
        kind="bar", figsize=(12, 5), rot=35
    )
    ax.set_title("Exit Mix by Fiscal Year")
    ax.set_ylabel("Share of move-out records")
    ax.legend(["Negative", "Transfer", "Evicted", "Positive"])
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart: matplotlib is not installed.")


## Question 3. What factors are associated with negative program exits, such as evictions?


In [ ]:
negative_reasons = (
    df[df["Resident Outcome"].eq("Negative")]
    .groupby(["HouseholdMoveOutReason", "HouseholdMoveOutCategory", "HouseholdMoveOutCategoryDetail"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

negative_by_fiscal_year = rate_table(df, "FiscalYear")
negative_by_property = rate_table(df, "DevelopmentName", min_records=50)
negative_by_family_type = rate_table(df, "HouseholdFamilyTypeHud")

display(negative_reasons.head(20))
display(negative_by_fiscal_year)
display(negative_by_property.head(20))
display(negative_by_family_type)


In [ ]:
# Criminal/drug-related trend.
criminal_drug_trend = (
    df.groupby("FiscalYear")
    .agg(
        total_records=("HouseholdEntityId", "count"),
        criminal_drug_related=("criminal_drug_related", "sum"),
    )
    .reset_index()
)
criminal_drug_trend["criminal_drug_rate"] = criminal_drug_trend["criminal_drug_related"] / criminal_drug_trend["total_records"]
criminal_drug_trend


## Question 4. What factors are associated with voluntary exits?


In [ ]:
voluntary_records = df[df["HouseholdMoveOutCategory"].eq("Voluntary Termination")].copy()

voluntary_reasons = (
    voluntary_records.groupby(["HouseholdMoveOutCategoryDetail", "Resident Outcome"], dropna=False)
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

voluntary_by_year = (
    df.groupby("FiscalYear")
    .agg(
        total_records=("HouseholdEntityId", "count"),
        voluntary_exits=("voluntary_exit", "sum"),
    )
    .reset_index()
)
voluntary_by_year["voluntary_exit_rate"] = voluntary_by_year["voluntary_exits"] / voluntary_by_year["total_records"]

voluntary_by_property = rate_table(df[df["voluntary_exit"].eq(1)], "DevelopmentName", min_records=20)

display(voluntary_reasons)
display(voluntary_by_year)
display(voluntary_by_property.head(20))


In [ ]:
# Optional voluntary-exit model.
if SKLEARN_AVAILABLE:
    X = model_df[predictors]
    y = model_df["voluntary_exit"]

    voluntary_model = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=2000)),
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    voluntary_model.fit(X_train, y_train)
    y_pred = voluntary_model.predict(X_test)
    y_prob = voluntary_model.predict_proba(X_test)[:, 1]

    print(classification_report(y_test, y_pred))
    print("AUC:", roc_auc_score(y_test, y_prob))
else:
    print("Skipping model: scikit-learn is not installed.")


## Question 5. Are there trends with move-outs and transfers at the property level?

This dataset supports property-level counts and shares among move-out records. To calculate true property-level rates, you still need property occupancy counts or households served by year.


In [ ]:
property_year_trends = (
    df.groupby(["DevelopmentName", "FiscalYear"])
    .agg(
        total_moveouts=("HouseholdEntityId", "count"),
        transfers=("transfer_exit", "sum"),
        negative_exits=("negative_exit", "sum"),
        positive_exits=("positive_exit", "sum"),
        evictions=("eviction_exit", "sum"),
    )
    .reset_index()
)

property_year_trends["transfer_share"] = property_year_trends["transfers"] / property_year_trends["total_moveouts"]
property_year_trends["negative_exit_share"] = property_year_trends["negative_exits"] / property_year_trends["total_moveouts"]
property_year_trends["positive_exit_share"] = property_year_trends["positive_exits"] / property_year_trends["total_moveouts"]

property_overall = (
    df.groupby("DevelopmentName")
    .agg(
        total_moveouts=("HouseholdEntityId", "count"),
        transfers=("transfer_exit", "sum"),
        negative_exits=("negative_exit", "sum"),
        positive_exits=("positive_exit", "sum"),
        evictions=("eviction_exit", "sum"),
    )
    .reset_index()
)
property_overall["transfer_share"] = property_overall["transfers"] / property_overall["total_moveouts"]
property_overall["negative_exit_share"] = property_overall["negative_exits"] / property_overall["total_moveouts"]
property_overall["positive_exit_share"] = property_overall["positive_exits"] / property_overall["total_moveouts"]

min_property_records = 50
property_overall_filtered = property_overall[property_overall["total_moveouts"] >= min_property_records].copy()

display(property_overall.sort_values("total_moveouts", ascending=False).head(20))
display(property_overall_filtered.sort_values("negative_exit_share", ascending=False).head(20))
display(property_overall_filtered.sort_values("transfer_share", ascending=False).head(20))


## Question 6. What recommendations would you make for increasing positive exits, especially self-sufficiency?


In [ ]:
positive_reasons = (
    df[df["Resident Outcome"].eq("Positive")]
    .groupby("HouseholdMoveOutReason")
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

self_sufficiency_records = df[df["self_sufficiency_exit"].eq(1)].copy()

positive_by_property = (
    df.groupby("DevelopmentName")
    .agg(
        total_records=("HouseholdEntityId", "count"),
        positive_exits=("positive_exit", "sum"),
        self_sufficiency_exits=("self_sufficiency_exit", "sum"),
    )
    .reset_index()
)
positive_by_property["positive_exit_rate"] = positive_by_property["positive_exits"] / positive_by_property["total_records"]
positive_by_property["self_sufficiency_exit_rate"] = positive_by_property["self_sufficiency_exits"] / positive_by_property["total_records"]

positive_by_property_filtered = positive_by_property[positive_by_property["total_records"] >= 50].copy()

display(positive_reasons)
print("Self-sufficiency records:", len(self_sufficiency_records))
display(self_sufficiency_records[[
    "HouseholdEntityId",
    "DevelopmentName",
    "HouseholdFamilyTypeHud",
    "HOHAgeAtMoveOut",
    "HouseholdRentToCharge",
    "HouseholdUnitTenure",
    "HouseholdProgramTenure",
    "FiscalYear",
    "HouseholdMoveOutReason",
]].head(50))
display(positive_by_property_filtered.sort_values("positive_exit_rate", ascending=False).head(20))


## Graphs and Visualizations

This section creates presentation-ready graphs from the analysis tables above. Run the notebook from the top before running these cells so the required dataframes already exist.

Graphs are saved as PNG files in:

`OUTPUT_DIR / "graphs"`


In [ ]:
GRAPH_DIR = OUTPUT_DIR / "graphs"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

if plt is None:
    print("matplotlib is not installed. Install matplotlib or run this notebook in Colab to generate graphs.")
else:
    print("Graphs will be saved to:", GRAPH_DIR)


In [ ]:
# Graph 1: Sentiment / resident outcome distribution
if plt is not None:
    plot_data = (
        df["Resident Outcome"]
        .fillna("Unknown")
        .value_counts()
        .reindex(["Negative", "Neutral", "Positive", "Unknown"])
        .dropna()
    )

    colors = {
        "Negative": "#C62828",
        "Neutral": "#F9A825",
        "Positive": "#2E7D32",
        "Unknown": "#78909C",
    }

    ax = plot_data.plot(
        kind="bar",
        figsize=(8, 5),
        color=[colors.get(x, "#607D8B") for x in plot_data.index],
    )
    ax.set_title("Move-Out Records by Sentiment")
    ax.set_xlabel("Sentiment")
    ax.set_ylabel("Number of records")
    ax.bar_label(ax.containers[0], fmt="%.0f")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "01_sentiment_distribution.png", dpi=200)
    plt.show()


In [ ]:
# Graph 2: Sentiment mix by fiscal year
if plt is not None:
    outcome_by_year = pd.crosstab(
        df["FiscalYear"],
        df["Resident Outcome"].fillna("Unknown")
    ).reindex(columns=["Negative", "Neutral", "Positive", "Unknown"], fill_value=0)

    ax = outcome_by_year.plot(
        kind="bar",
        stacked=True,
        figsize=(11, 6),
        color=["#C62828", "#F9A825", "#2E7D32", "#78909C"],
    )
    ax.set_title("Move-Out Sentiment by Fiscal Year")
    ax.set_xlabel("Fiscal Year")
    ax.set_ylabel("Number of records")
    plt.xticks(rotation=35, ha="right")
    plt.legend(title="Sentiment", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "02_sentiment_by_fiscal_year.png", dpi=200)
    plt.show()


In [ ]:
# Graph 3: Negative exit rate and transfer share by fiscal year
if plt is not None:
    plot_df = covid_summary.set_index("FiscalYear")

    ax = plot_df[["negative_rate", "transfer_rate", "evictions_rate", "positive_rate"]].plot(
        kind="line",
        marker="o",
        figsize=(11, 6),
        color=["#C62828", "#1565C0", "#EF6C00", "#2E7D32"],
    )
    ax.set_title("Exit Pattern Rates by Fiscal Year")
    ax.set_xlabel("Fiscal Year")
    ax.set_ylabel("Share of move-out records")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    plt.xticks(rotation=35, ha="right")
    plt.grid(True, alpha=0.25)
    plt.legend(["Negative", "Transfer", "Evicted", "Positive"])
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "03_exit_pattern_rates_by_fiscal_year.png", dpi=200)
    plt.show()


In [ ]:
# Graph 4: Top negative move-out reasons
if plt is not None:
    top_negative = negative_reasons.head(10).sort_values("records")

    ax = top_negative.plot(
        kind="barh",
        x="HouseholdMoveOutCategoryDetail",
        y="records",
        figsize=(10, 6),
        legend=False,
        color="#C62828",
    )
    ax.set_title("Top 10 Negative Move-Out Reasons")
    ax.set_xlabel("Number of records")
    ax.set_ylabel("Move-out reason detail")
    ax.bar_label(ax.containers[0], fmt="%.0f")
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "04_top_negative_move_out_reasons.png", dpi=200)
    plt.show()


In [ ]:
# Graph 5: Negative exit rate by household type
if plt is not None:
    plot_data = negative_by_household_type.sort_values("negative_exit_rate", ascending=True)

    ax = plot_data.plot(
        kind="barh",
        x="HouseholdFamilyTypeHud",
        y="negative_exit_rate",
        figsize=(8, 5),
        legend=False,
        color="#8E2430",
    )
    ax.set_title("Negative Exit Rate by Household Type")
    ax.set_xlabel("Negative exit rate")
    ax.set_ylabel("Household type")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "05_negative_exit_rate_by_household_type.png", dpi=200)
    plt.show()


In [ ]:
# Graph 6: Highest negative exit rate properties, among properties with at least 50 records
if plt is not None:
    top_properties = (
        property_overall_filtered
        .sort_values("negative_exit_share", ascending=False)
        .head(15)
        .sort_values("negative_exit_share", ascending=True)
    )

    ax = top_properties.plot(
        kind="barh",
        x="DevelopmentName",
        y="negative_exit_share",
        figsize=(10, 7),
        legend=False,
        color="#C62828",
    )
    ax.set_title("Top Properties by Negative Exit Share")
    ax.set_xlabel("Negative exit share")
    ax.set_ylabel("Development")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "06_top_properties_negative_exit_share.png", dpi=200)
    plt.show()


In [ ]:
# Graph 7: Highest transfer-share properties, among properties with at least 50 records
if plt is not None:
    top_transfer_properties = (
        property_overall_filtered
        .sort_values("transfer_share", ascending=False)
        .head(15)
        .sort_values("transfer_share", ascending=True)
    )

    ax = top_transfer_properties.plot(
        kind="barh",
        x="DevelopmentName",
        y="transfer_share",
        figsize=(10, 7),
        legend=False,
        color="#1565C0",
    )
    ax.set_title("Top Properties by Transfer Share")
    ax.set_xlabel("Transfer share")
    ax.set_ylabel("Development")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "07_top_properties_transfer_share.png", dpi=200)
    plt.show()


In [ ]:
# Graph 8: Positive exit reasons
if plt is not None:
    plot_data = positive_reasons.sort_values("records")

    ax = plot_data.plot(
        kind="barh",
        x="HouseholdMoveOutReason",
        y="records",
        figsize=(10, 5),
        legend=False,
        color="#2E7D32",
    )
    ax.set_title("Positive Exit Reasons")
    ax.set_xlabel("Number of records")
    ax.set_ylabel("Move-out reason")
    ax.bar_label(ax.containers[0], fmt="%.0f")
    plt.tight_layout()
    plt.savefig(GRAPH_DIR / "08_positive_exit_reasons.png", dpi=200)
    plt.show()


In [ ]:
# List graph files generated
if plt is not None:
    for path in sorted(GRAPH_DIR.glob("*.png")):
        print(path)


## Export Results


In [ ]:
outputs = {
    "outcome_summary.csv": outcome_summary,
    "category_summary.csv": category_summary,
    "missing_summary.csv": missing_summary,
    "negative_by_household_type.csv": negative_by_household_type,
    "negative_by_race.csv": negative_by_race,
    "negative_by_ethnicity.csv": negative_by_ethnicity,
    "negative_by_property.csv": negative_by_property,
    "covid_exit_summary.csv": covid_summary,
    "negative_reasons.csv": negative_reasons,
    "criminal_drug_trend.csv": criminal_drug_trend,
    "voluntary_reasons.csv": voluntary_reasons,
    "voluntary_by_year.csv": voluntary_by_year,
    "property_year_trends.csv": property_year_trends,
    "property_overall.csv": property_overall,
    "positive_reasons.csv": positive_reasons,
    "positive_by_property.csv": positive_by_property,
    "self_sufficiency_records.csv": self_sufficiency_records,
}

if SKLEARN_AVAILABLE and not negative_exit_coefficients.empty:
    outputs["negative_exit_model_coefficients.csv"] = negative_exit_coefficients

for filename, table in outputs.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)

metadata = {
    "activity_file": str(ACTIVITY_PATH),
    "reasons_file": str(REASONS_PATH),
    "schema_file": str(SCHEMA_PATH),
    "rows": int(len(df)),
    "known_outcome_rows": int(df["Resident Outcome"].notna().sum()),
    "complete_rows_original_fields": int(complete_rows),
    "important_limitation": "The dataset contains move-out/transfer records only, not the full active household population.",
}

with open(OUTPUT_DIR / "analysis_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved analysis outputs to:", OUTPUT_DIR)


## Final Notes for Interpretation

- This notebook answers the six questions using the current move-out activity data and move-out reason dictionary.
- For true move-out risk modeling, request active household records for households who did not move out.
- For stronger COVID analysis, request continuous pre-COVID years from FY 2015-2016 through FY 2019-2020.
- For property-level rates, request property occupancy counts or households served by year.
- For self-sufficiency recommendations, request income, employment, FSS, case management, service participation, and return-to-assistance data.
